# Lumis 04: Calibration and confidence bands

Answers RQ2: is a calibrated confidence reliable enough to decide when Lumis should defer to a professional? (CP1 sections 3.4.2 and 3.5.1)

1. Measures how well the raw model confidence matches reality (Expected Calibration Error, reliability diagram)
2. Fits **temperature scaling** on the validation set. The calibrated scores reported here are out-of-fold (5-fold cross-validation inside the validation set), so the model is never graded on the data its temperature was fitted on
3. **Selective accuracy:** how accurate Lumis is when it only answers cases above a confidence threshold, and how many cases it refers
4. Sets the **confidence bands** the app shows (high, moderate, low = refer)
5. **Monte Carlo dropout** on the classifier head, to check whether it flags errors better than calibrated confidence alone

Uses the saved validation logits from notebook 03; the test set is still untouched. **T4 GPU** recommended (step 5 runs the model).

## 1. Settings and the run to calibrate

By default this picks the run with the best validation macro F1 from `runs_log.csv`. To calibrate a specific run, set `RUN` to its name.

In [ ]:
# Chosen model: EfficientNet-B0. Its macro F1 ties ResNet50 (0.652 vs 0.656, within noise on 998 images),
# but it has higher macro sensitivity (68% vs 66%), far better scabies sensitivity (69% vs 49%),
# and 6x fewer parameters, which matters for the 5-second response target (NFR1).
RUN = 'efficientnetb0_20261108_2120'   # None = best macro F1 in runs_log.csv
TARGET_HIGH = 0.85     # accuracy required inside the 'high confidence' band
TARGET_ANSWERED = 0.75 # accuracy required across everything Lumis answers (high + moderate)
MC_PASSES = 30
SEED = 42

from google.colab import drive
drive.mount('/content/drive')

import os, json, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

BASE = '/content/drive/MyDrive/Lumis'
runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if RUN is None:
    RUN = runs.sort_values('val_macro_f1').iloc[-1]['run']
info = runs[runs.run == RUN].iloc[-1]
RUN_DIR = f'{BASE}/exports/runs/{RUN}'
logits = np.load(f'{RUN_DIR}/val_logits.npy')
y = np.load(f'{RUN_DIR}/val_labels.npy')
print(f'Calibrating {RUN} ({info.backbone}), validation macro F1 {info.val_macro_f1}, {len(y)} images')
display(runs[['run', 'backbone', 'val_accuracy', 'val_macro_f1']])

## 2. Calibration tools

In [ ]:
from scipy.optimize import minimize_scalar

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def ece(probs, y, bins=15):
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges, total = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)

def nll(probs, y):
    return float(-np.mean(np.log(probs[np.arange(len(y)), y] + 1e-12)))

def fit_temperature(z, y):
    res = minimize_scalar(lambda lt: nll(softmax(z / np.exp(lt)), y), bounds=(-3, 3), method='bounded')
    return float(np.exp(res.x))

def reliability(ax, probs, y, title, bins=10):
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges = np.linspace(0, 1, bins + 1)
    mids, accs, counts = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            mids.append((lo + hi) / 2); accs.append(correct[m].mean()); counts.append(m.sum())
    ax.bar(mids, accs, width=1 / bins, edgecolor='black', alpha=0.7, label='accuracy in bin')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect calibration')
    ax.set_xlabel('model confidence'); ax.set_ylabel('actual accuracy')
    ax.set_title(f'{title}\nECE = {ece(probs, y):.3f}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc='upper left', fontsize=8)

## 3. Temperature scaling

One number, T, divides the logits. T above 1 means the raw model was overconfident.

In [ ]:
from sklearn.model_selection import StratifiedKFold

raw = softmax(logits)
oof = np.zeros_like(raw)
fold_T = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(logits, y):
    T = fit_temperature(logits[tr], y[tr])
    fold_T.append(T)
    oof[te] = softmax(logits[te] / T)

T_FINAL = fit_temperature(logits, y)   # fitted on all validation data: this is what the app uses
cal_stats = {
    'temperature': round(T_FINAL, 4),
    'temperature_per_fold': [round(t, 3) for t in fold_T],
    'ece_raw': round(ece(raw, y), 4), 'ece_calibrated_oof': round(ece(oof, y), 4),
    'nll_raw': round(nll(raw, y), 4), 'nll_calibrated_oof': round(nll(oof, y), 4),
}
print(json.dumps(cal_stats, indent=1))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
reliability(ax[0], raw, y, 'Before: raw softmax')
reliability(ax[1], oof, y, f'After: temperature scaling (T = {T_FINAL:.2f}, out-of-fold)')
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/reliability_diagram.png', dpi=100)
plt.show()

## 4. Selective accuracy and confidence bands

If Lumis only answers when its calibrated confidence is at least a threshold, how accurate are those answers, and what share of cases does it refer? The bands use the lowest threshold that meets each target accuracy.

In [ ]:
conf = oof.max(1)
correct = oof.argmax(1) == y
n = len(y)

order = np.argsort(-conf)
coverage = np.arange(1, n + 1) / n
acc_at = np.cumsum(correct[order]) / np.arange(1, n + 1)

rows = []
for t in np.round(np.arange(0.3, 0.95, 0.05), 2):
    m = conf >= t
    rows.append({'threshold': t, 'answered': f'{m.mean():.0%}', 'referred': f'{1 - m.mean():.0%}',
                 'accuracy_when_answering': round(correct[m].mean(), 3) if m.any() else None})
sel = pd.DataFrame(rows)
display(sel)

def lowest_threshold(target, min_cases=20):
    for t in np.round(np.arange(0.2, 0.99, 0.01), 2):
        m = conf >= t
        if m.sum() >= min_cases and correct[m].mean() >= target:
            return float(t)
    return None

t_high = lowest_threshold(TARGET_HIGH)
t_low = lowest_threshold(TARGET_ANSWERED)
bands = {'high_from': t_high, 'moderate_from': t_low, 'refer_below': t_low}
if t_high and t_low:
    hi, mid, lo = conf >= t_high, (conf >= t_low) & (conf < t_high), conf < t_low
    bands['share'] = {'high': round(hi.mean(), 3), 'moderate': round(mid.mean(), 3), 'refer': round(lo.mean(), 3)}
    bands['accuracy'] = {'high': round(correct[hi].mean(), 3),
                         'moderate': round(correct[mid].mean(), 3) if mid.any() else None,
                         'refer': round(correct[lo].mean(), 3) if lo.any() else None}
print(json.dumps(bands, indent=1))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(coverage, acc_at, lw=2)
ax.axhline(correct.mean(), color='gray', ls=':', lw=1, label=f'answer everything ({correct.mean():.0%})')
for t, lab in [(t_low, 'refer below'), (t_high, 'high band from')]:
    if t:
        c = (conf >= t).mean()
        ax.axvline(c, color='C1', ls='--', lw=1)
        ax.text(c, 0.42, f' {lab} {t:.2f}', color='C1', fontsize=8, rotation=90, va='bottom')
ax.set_xlabel('share of cases Lumis answers (the rest are referred)')
ax.set_ylabel('accuracy on answered cases')
ax.set_title('Selective accuracy (calibrated, out-of-fold)')
ax.set_ylim(0.4, 1.0); ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/selective_accuracy.png', dpi=100)
plt.show()

## 5. Monte Carlo dropout on the classifier head

The backbone features are computed once; the head's dropout is then kept switched on for repeated passes. The spread of those predictions (mutual information) estimates the model's own uncertainty. This is a lightweight, last-layer version of MC dropout.

The check: does MC uncertainty flag wrong answers better than calibrated confidence alone? (AUROC for detecting errors; 0.5 = no better than chance.)

In [ ]:
import tensorflow as tf
import keras
from sklearn.metrics import roc_auc_score

LOCAL = '/content/dermnet'
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)

PREPROCESS = {'mobilenetv2': keras.applications.mobilenet_v2.preprocess_input,
              'efficientnetb0': keras.applications.efficientnet.preprocess_input,
              'resnet50': keras.applications.resnet50.preprocess_input}[info.backbone]

model = keras.models.load_model(info.model_file)
gap = [l for l in model.layers if isinstance(l, keras.layers.GlobalAveragePooling2D)][0]
rate = [l for l in model.layers if isinstance(l, keras.layers.Dropout)][0].rate
W, b = model.get_layer('logits').get_weights()
features_model = keras.Model(model.inputs, gap.output)

split = pd.read_csv(f'{BASE}/data/lumis_split_v1.csv')
val_df = split[split.split == 'val'].reset_index(drop=True)
size = int(info.img_size)

def load(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.cast(tf.cast(tf.image.resize(img, (size, size)), tf.uint8), tf.float32)  # same rounding as notebook 03
    return PREPROCESS(img)

ds = (tf.data.Dataset.from_tensor_slices([os.path.join(LOCAL, p) for p in val_df.path])
      .map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(32))
feats = features_model.predict(ds, verbose=0)

check = feats @ W + b
print('Features reproduce the saved logits:', np.allclose(check, logits, atol=1e-2))

rng = np.random.default_rng(SEED)
passes = []
for _ in range(MC_PASSES):
    mask = rng.random(feats.shape) >= rate
    passes.append(softmax(((feats * mask / (1 - rate)) @ W + b) / T_FINAL))
passes = np.stack(passes)
mean_p = passes.mean(0)

entropy = lambda p: -(p * np.log(p + 1e-12)).sum(-1)
mutual_info = entropy(mean_p) - entropy(passes).mean(0)
mc_correct = mean_p.argmax(1) == y
calibrated = softmax(logits / T_FINAL)

mc_stats = {
    'mc_passes': MC_PASSES, 'dropout_rate': rate,
    'accuracy_single_pass': round(float((calibrated.argmax(1) == y).mean()), 4),
    'accuracy_mc_mean': round(float(mc_correct.mean()), 4),
    'error_auroc_calibrated_confidence': round(float(roc_auc_score(~mc_correct, -mean_p.max(1))), 4),
    'error_auroc_mc_mutual_information': round(float(roc_auc_score(~mc_correct, mutual_info)), 4),
}
print(json.dumps(mc_stats, indent=1))

## 6. Save the calibration for the app

In [ ]:
out = {'run': RUN, 'backbone': info.backbone, 'classes': sorted(split.lumis_class.unique()),
       'targets': {'high': TARGET_HIGH, 'answered': TARGET_ANSWERED},
       **cal_stats, 'bands': bands, 'mc_dropout': mc_stats}
json.dump(out, open(f'{RUN_DIR}/calibration.json', 'w'), indent=1)
sel.to_csv(f'{RUN_DIR}/selective_accuracy.csv', index=False)
print('Saved to', RUN_DIR)
print(json.dumps(out, indent=1))